# Data Cleaning Pipeline

This notebook cleans the raw data from `track1_dataset_notes.txt` guidelines.
It normalizes IDs, dates, financial amounts, and categorical fields.

In [ ]:
# ============================================================
# CELL 1: Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

# Paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
# ============================================================
# CELL 2: LOAD RAW DATASETS
# ============================================================

transactions = pd.read_csv(RAW_DIR / "track1_upi_transactions.csv")
kyc = pd.read_csv(RAW_DIR / "track1_kyc_records.csv")
merchants = pd.read_csv(RAW_DIR / "track1_merchants_master.csv")
with open(RAW_DIR / "track1_chargebacks.json", "r", encoding="utf-8") as f:
    chargebacks = pd.DataFrame(json.load(f))

In [19]:
print("Transactions :", len(transactions))
print("KYC          :", len(kyc))
print("Merchants    :", len(merchants))
print("Chargebacks  :", len(chargebacks))

Transactions : 20400
KYC          : 36400
Merchants    : 6210
Chargebacks  : 2884


## 2. Standardize Identifiers

Normalizing `user_id` (USR12345), `merchant_id` (MCH1234), `txn_id` (TXN12345), and `complaint_id` (CBK12345).

In [ ]:
# ============================================================
# CELL 3 : CLEAN DATASETS
# ============================================================

def clean_id(val, prefix):
    if pd.isna(val):
        return val
    # Remove all non-alphanumeric chars and convert to upper
    val = re.sub(r'[^a-zA-Z0-9]', '', str(val)).upper()
    # Remove prefix if it already exists to avoid duplication
    if val.startswith(prefix):
        val = val[len(prefix):]
    # Some IDs may just be numbers initially
    return f"{prefix}{val}"

# Clean Transactions
transactions['user_id'] = transactions['user_id'].apply(lambda x: clean_id(x, 'USR'))
transactions['merchant_id'] = transactions['merchant_id'].apply(lambda x: clean_id(x, 'MCH'))
transactions['txn_id'] = transactions['txn_id'].apply(lambda x: clean_id(x, 'TXN'))

# Clean KYC
kyc['user_id'] = kyc['user_id'].apply(lambda x: clean_id(x, 'USR'))

# Clean Merchants
merchants['merchant_id'] = merchants['merchant_id'].apply(lambda x: clean_id(x, 'MCH'))

# Clean Chargebacks
chargebacks['complaint_id'] = chargebacks['complaint_id'].apply(lambda x: clean_id(x, 'CBK'))
chargebacks['txn_id'] = chargebacks['txn_id'].apply(lambda x: clean_id(x, 'TXN'))
chargebacks['user_id'] = chargebacks['user_id'].apply(lambda x: clean_id(x, 'USR'))
chargebacks['merchant_id'] = chargebacks['merchant_id'].apply(lambda x: clean_id(x, 'MCH'))

## 3. Clean Financial Amounts

Handles currencies, commas, blanks, and negative values.

In [21]:
# ============================================================
# CELL 4: CLEAN MONETARY VALUES
# ============================================================

def clean_amount(val):
    """
    Convert messy monetary values into numeric values.

    Handles:
    - Currency symbols: ₹, Rs., INR
    - Commas: 16,466.93
    - K notation: 27.3k
    - Whitespace
    - Missing values
    - Negative values

    Negative values are preserved initially so that
    invalid business values can be flagged separately.
    """

    if pd.isna(val):
        return np.nan

    val = str(val).strip()

    # Handle textual missing values
    if val.upper() in {
        "", "NA", "N/A", "NONE", "NULL",
        "NAN", "NOT AVAILABLE", "NOT_AVAILABLE"
    }:
        return np.nan

    # Remove currency indicators
    val = re.sub(r"(?i)INR|RS\.?|₹", "", val)

    # Remove commas
    val = val.replace(",", "").strip()

    # Handle K notation
    multiplier = 1

    if re.fullmatch(r"[-+]?\d*\.?\d+\s*[Kk]", val):
        multiplier = 1000
        val = re.sub(r"[Kk]", "", val).strip()

    try:
        return float(val) * multiplier

    except (ValueError, TypeError):
        return np.nan


# ------------------------------------------------------------
# Apply monetary cleaning
# ------------------------------------------------------------

transactions["amount"] = transactions["amount"].apply(clean_amount)

kyc["monthly_income"] = kyc["monthly_income"].apply(clean_amount)

merchants["declared_avg_ticket_size"] = (
    merchants["declared_avg_ticket_size"].apply(clean_amount)
)

chargebacks["disputed_amount"] = (
    chargebacks["disputed_amount"].apply(clean_amount)
)


# ------------------------------------------------------------
# Create business-rule validation flags
# ------------------------------------------------------------

transactions["amount_invalid"] = (
    transactions["amount"].notna() &
    (transactions["amount"] <= 0)
)

kyc["income_invalid"] = (
    kyc["monthly_income"].notna() &
    (kyc["monthly_income"] <= 0)
)

merchants["ticket_size_invalid"] = (
    merchants["declared_avg_ticket_size"].notna() &
    (merchants["declared_avg_ticket_size"] <= 0)
)

chargebacks["disputed_amount_invalid"] = (
    chargebacks["disputed_amount"].notna() &
    (chargebacks["disputed_amount"] <= 0)
)


# ------------------------------------------------------------
# Convert invalid business values to NaN
# ------------------------------------------------------------

transactions.loc[
    transactions["amount_invalid"], "amount"
] = np.nan

kyc.loc[
    kyc["income_invalid"], "monthly_income"
] = np.nan

merchants.loc[
    merchants["ticket_size_invalid"],
    "declared_avg_ticket_size"
] = np.nan

chargebacks.loc[
    chargebacks["disputed_amount_invalid"],
    "disputed_amount"
] = np.nan


print("Monetary cleaning completed.")
print("Invalid transaction amounts:",
      transactions["amount_invalid"].sum())
print("Invalid KYC incomes:",
      kyc["income_invalid"].sum())
print("Invalid merchant ticket sizes:",
      merchants["ticket_size_invalid"].sum())
print("Invalid chargeback amounts:",
      chargebacks["disputed_amount_invalid"].sum())

Monetary cleaning completed.
Invalid transaction amounts: 429
Invalid KYC incomes: 1456
Invalid merchant ticket sizes: 501
Invalid chargeback amounts: 231


## 4. Standardize Timestamps

Parsing mixed formats including Unix epoch times.

In [ ]:
# ============================================================
# CELL 5 : CLEAN TIMESTAMPS
# ============================================================

def clean_timestamp(val):
    if pd.isna(val) or str(val).strip() == "":
        return pd.NaT
    val = str(val).strip()
    
    # Check Unix timestamps
    if val.isdigit():
        if len(val) == 10:
            return pd.to_datetime(int(val), unit='s')
        elif len(val) >= 13:
            return pd.to_datetime(int(val[:13]), unit='ms')
            
    try:
        return pd.to_datetime(val, errors='coerce', format='mixed')
    except:
        return pd.NaT

transactions['timestamp'] = transactions['timestamp'].apply(clean_timestamp)
kyc['date_of_birth'] = kyc['date_of_birth'].apply(clean_timestamp).dt.date
kyc['signup_timestamp'] = kyc['signup_timestamp'].apply(clean_timestamp)
merchants['onboarding_date'] = merchants['onboarding_date'].apply(clean_timestamp).dt.date
chargebacks['transaction_timestamp'] = chargebacks['transaction_timestamp'].apply(clean_timestamp)
chargebacks['reported_timestamp'] = chargebacks['reported_timestamp'].apply(clean_timestamp)
chargebacks['bank_response_timestamp'] = chargebacks['bank_response_timestamp'].apply(clean_timestamp)

In [23]:
# ============================================================
# CELL 5A: TIMELINE ANOMALY FLAGS
# ============================================================

# Chargeback timeline validation
chargebacks["reported_before_transaction"] = (
    chargebacks["reported_timestamp"].notna() &
    chargebacks["transaction_timestamp"].notna() &
    (chargebacks["reported_timestamp"] < chargebacks["transaction_timestamp"])
)

chargebacks["bank_response_before_reported"] = (
    chargebacks["bank_response_timestamp"].notna() &
    chargebacks["reported_timestamp"].notna() &
    (
        chargebacks["bank_response_timestamp"]
        < chargebacks["reported_timestamp"]
    )
)

print(
    "Reported before transaction:",
    chargebacks["reported_before_transaction"].sum()
)

print(
    "Bank response before reported:",
    chargebacks["bank_response_before_reported"].sum()
)

Reported before transaction: 351
Bank response before reported: 246


## 5. Normalize Categorical Data

Consolidating varying status fields, severity, and text fields like city/state.

In [ ]:
# ============================================================
# CELL 6 : CLEAN CATEGORICAL COLUMNS
# ============================================================

def clean_status(val, status_map):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().upper()
    return status_map.get(val, val)

txn_status_map = {
    'SUCCESS': 'SUCCESS', 'TXN_SUCCESS': 'SUCCESS', 'S': 'SUCCESS', 'COMPLETED': 'SUCCESS',
    'FAILED': 'FAILED', 'TXN_FAILED': 'FAILED', 'F': 'FAILED', 'FAIL': 'FAILED', 'DECLINED': 'FAILED',
    'PENDING': 'PENDING', 'P': 'PENDING', 'PROCESSING': 'PENDING', 'INITIATED': 'PENDING'
}
transactions['status'] = transactions['status'].apply(lambda x: clean_status(x, txn_status_map))

kyc_status_map = {
    'VERIFIED': 'VERIFIED', 'V': 'VERIFIED', 'DONE': 'VERIFIED', 'APPROVED': 'VERIFIED', 'KYC_DONE': 'VERIFIED',
    'REJECTED': 'REJECTED', 'REJECT': 'REJECTED', 'R': 'REJECTED', 'FAILED': 'REJECTED',
    'PENDING': 'PENDING', 'P': 'PENDING', 'IN_PROGRESS': 'PENDING', 'UNDER REVIEW': 'PENDING'
}
kyc['kyc_status'] = kyc['kyc_status'].apply(lambda x: clean_status(x, kyc_status_map))

kyc_risk_map = {
    'LOW': 'LOW', 'L': 'LOW',
    'MEDIUM': 'MEDIUM', 'M': 'MEDIUM',
    'HIGH': 'HIGH', 'H': 'HIGH', 'CRITICAL': 'HIGH', 'CRIT': 'HIGH',
    'UNKNOWN': 'UNKNOWN'
}
kyc['risk_segment'] = kyc['risk_segment'].apply(lambda x: clean_status(x, kyc_risk_map))

cbk_status_map = {
    'OPEN': 'OPEN', 'WIP': 'OPEN', 'IN_PROGRESS': 'OPEN', 'IN PROGRESS': 'OPEN', 'PENDING BANK': 'OPEN', 'PENDING_BANK': 'OPEN',
    'CLOSED': 'CLOSED', 'RESOLVED': 'CLOSED',
    'REJECTED': 'REJECTED'
}
chargebacks['resolution_status'] = chargebacks['resolution_status'].apply(lambda x: clean_status(x, cbk_status_map))

merchant_status_map = {
    'ACTIVE': 'ACTIVE', 'A': 'ACTIVE', 'LIVE': 'ACTIVE', 'ENABLED': 'ACTIVE',
    'INACTIVE': 'INACTIVE', 'I': 'INACTIVE', 'CLOSED': 'INACTIVE',
    'SUSPENDED': 'SUSPENDED', 'HOLD': 'SUSPENDED', 'DISABLED': 'SUSPENDED', 'BLOCKED': 'SUSPENDED', 'S': 'SUSPENDED'
}
merchants['merchant_status'] = merchants['merchant_status'].apply(lambda x: clean_status(x, merchant_status_map))

cbk_severity_map = {
    'LOW': 'LOW', 'L': 'LOW', 'P4': 'LOW',
    'MEDIUM': 'MEDIUM', 'M': 'MEDIUM', 'P3': 'MEDIUM',
    'HIGH': 'HIGH', 'H': 'HIGH', 'P2': 'HIGH',
    'CRITICAL': 'CRITICAL', 'CRIT': 'CRITICAL', 'P1': 'CRITICAL'
}
chargebacks['severity'] = chargebacks['severity'].apply(lambda x: clean_status(x, cbk_severity_map))

# Name Cleaning
def clean_name(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    val = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', val)
    return val.title()

kyc['full_name'] = kyc['full_name'].apply(clean_name)
merchants['merchant_name'] = merchants['merchant_name'].apply(clean_name)

# Chargebacks text cleaning
chargebacks['channel'] = chargebacks['channel'].str.upper().str.strip()
chargebacks['reason_code'] = chargebacks['reason_code'].str.replace('_', ' ').str.title().str.strip()

# City and State Cleaning
kyc['city'] = kyc['city'].str.title().str.strip()
kyc['state'] = kyc['state'].str.title().str.strip()
merchants['city'] = merchants['city'].str.title().str.strip()
merchants['state'] = merchants['state'].str.title().str.strip()

# Common abbreviations mapping
city_map = {'Blr': 'Bangalore', 'Lko': 'Lucknow', 'Jpr': 'Jaipur', 'Ldh': 'Ludhiana', 'Hyd': 'Hyderabad', 'Asr': 'Amritsar', 'Dilli': 'Delhi', 'Calcutta': 'Kolkata', 'Bombay': 'Mumbai', 'Mumbay': 'Mumbai'}
kyc['city'] = kyc['city'].replace(city_map)
merchants['city'] = merchants['city'].replace(city_map)

merchants['merchant_category'] = merchants['merchant_category'].str.replace('_', ' ').str.title().str.strip()
merchants['business_type'] = merchants['business_type'].str.replace('_', ' ').str.replace('-', ' ').str.title().str.strip()

# ============================================================
# CELL 6A: FINAL MERCHANT CATEGORY STANDARDIZATION
# ============================================================

def standardize_merchant_category(value):
    """
    Standardize merchant categories after title-case cleaning.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    category_map = {

        # Retail
        "DEPT STORE": "Department Store",
        "DEPARTMENT STORE": "Department Store",
        "RETAIL OTHER": "Retail",

        # Hospitality
        "HOTEL": "Hotel Lodging",
        "HOTELS": "Hotel Lodging",
        "HOTEL LODGING": "Hotel Lodging",

        # Telecom
        "PHONE SERVICE": "Telecom",
        "TELECOM": "Telecom",

        # Books / stationery
        "BOOKS": "Books Stationery",
        "STATIONERY": "Books Stationery",
        "BOOKS STATIONERY": "Books Stationery",

        # Clothing
        "CLOTHS": "Garments",
        "CLOTHES": "Garments",
        "GARMENTS": "Garments",

        # Medical
        "MEDICAL": "Medical Store",
        "MEDICAL STORE": "Medical Store",

        # Transport
        "TRANSPRT": "Transport",
        "TRANSPORT": "Transport",

        # Food
        "FOOD": "Restaurant",
        "RESTAURANT": "Restaurant"
    }

    return category_map.get(value, str(value).strip().title())


merchants["merchant_category"] = (
    merchants["merchant_category"]
    .apply(standardize_merchant_category)
)


print("Merchant categories standardized.")
print(
    merchants["merchant_category"]
    .value_counts()
    .head(20)
)

Merchant categories standardized.
merchant_category
Telecom              479
Hotel Lodging        477
Books Stationery     449
Department Store     333
Retail               325
Garments             295
Transport            279
Medical Store        264
Restaurant           247
Miscellaneous        159
Mobile Recharge      156
Hospitality          145
Book Store           144
Other                144
Misc Retail          143
Kirana               139
Department Stores    135
Transportation       126
Apparel              126
Pharmacy             125
Name: count, dtype: int64


## 6. Format Verification specific fields

UTR validation, PAN/Aadhaar basic cleanup, and MCC format normalization.

In [ ]:
# ============================================================
# CELL 7: CLEAN IDENTIFIERS
# ============================================================

def clean_utr(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().upper()
    val = re.sub(r'[^A-Z0-9]', '', val)
    if val == '':
        return np.nan
    if not val.startswith('UTR'):
        val = 'UTR' + val
    return val
transactions['utr'] = transactions['utr'].apply(clean_utr)

def clean_mcc(val):
    """
    Normalize Merchant Category Codes.

    Examples:
    MCC-7011 -> 7011
    5311.0  -> 5311
    05311   -> 5311
    MCC-5912 -> 5912

    Only valid 4-digit MCC values are retained.
    """

    if pd.isna(val):
        return np.nan

    val = str(val).strip().upper()

    # Remove MCC prefix
    val = re.sub(r"^MCC[-_\s]*", "", val)

    # Handle values such as 5311.0
    if re.fullmatch(r"\d+\.0", val):
        val = val[:-2]

    # Keep digits only
    val = re.sub(r"[^0-9]", "", val)

    # Accept only exactly 4 digits
    if len(val) == 4:
        return val

    # Some values have a leading zero but represent a 4-digit MCC
    if len(val) == 5 and val.startswith("0"):
        return val[1:]

    return np.nan


transactions["mcc"] = transactions["mcc"].apply(clean_mcc)

merchants["mcc"] = merchants["mcc"].apply(clean_mcc)

def clean_aadhaar(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r'[^0-9]', '', str(val))
    return val if len(val) == 12 else np.nan
kyc['aadhaar'] = kyc['aadhaar'].apply(clean_aadhaar)

def clean_pan(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r'[^A-Z0-9]', '', str(val).upper())
    return val if len(val) == 10 else np.nan
kyc['pan'] = kyc['pan'].apply(clean_pan)

def clean_settlement(val):
    if pd.isna(val):
        return np.nan
    return re.sub(r'[^A-Z0-9]', '', str(val).upper())
merchants['settlement_account'] = merchants['settlement_account'].apply(clean_settlement)

## 7. Handle Duplicates and Missing Primary Keys

In [26]:
# ============================================================
# CELL 8: DUPLICATE & MISSING-ID AUDIT
# ============================================================

# ------------------------------------------------------------
# Exact duplicate flags
# ------------------------------------------------------------

transactions["exact_duplicate"] = transactions.duplicated(keep=False)

kyc["exact_duplicate"] = kyc.duplicated(keep=False)

merchants["exact_duplicate"] = merchants.duplicated(keep=False)

chargebacks["exact_duplicate"] = chargebacks.duplicated(keep=False)


# ------------------------------------------------------------
# Missing ID flags
# ------------------------------------------------------------

transactions["missing_txn_id"] = transactions["txn_id"].isna()
transactions["missing_user_id"] = transactions["user_id"].isna()
transactions["missing_merchant_id"] = transactions["merchant_id"].isna()

kyc["missing_user_id"] = kyc["user_id"].isna()

merchants["missing_merchant_id"] = merchants["merchant_id"].isna()

chargebacks["missing_complaint_id"] = chargebacks["complaint_id"].isna()
chargebacks["missing_txn_id"] = chargebacks["txn_id"].isna()
chargebacks["missing_user_id"] = chargebacks["user_id"].isna()
chargebacks["missing_merchant_id"] = chargebacks["merchant_id"].isna()


# ------------------------------------------------------------
# Audit summary
# ------------------------------------------------------------

print("========== DUPLICATE AUDIT ==========")

print(
    "Transactions exact duplicate rows:",
    transactions["exact_duplicate"].sum()
)

print(
    "KYC exact duplicate rows:",
    kyc["exact_duplicate"].sum()
)

print(
    "Merchants exact duplicate rows:",
    merchants["exact_duplicate"].sum()
)

print(
    "Chargebacks exact duplicate rows:",
    chargebacks["exact_duplicate"].sum()
)

print("\n========== MISSING ID AUDIT ==========")

print(
    "Transactions missing IDs:",
    transactions[
        ["missing_txn_id", "missing_user_id", "missing_merchant_id"]
    ].sum()
)

print(
    "KYC missing user IDs:",
    kyc["missing_user_id"].sum()
)

print(
    "Merchant missing merchant IDs:",
    merchants["missing_merchant_id"].sum()
)

print(
    "Chargeback missing IDs:",
    chargebacks[
        [
            "missing_complaint_id",
            "missing_txn_id",
            "missing_user_id",
            "missing_merchant_id"
        ]
    ].sum()
)

========== DUPLICATE AUDIT ==========
Transactions exact duplicate rows: 800
KYC exact duplicate rows: 972
Merchants exact duplicate rows: 212
Chargebacks exact duplicate rows: 168

========== MISSING ID AUDIT ==========
Transactions missing IDs: missing_txn_id         0
missing_user_id        0
missing_merchant_id    0
dtype: int64
KYC missing user IDs: 0
Merchant missing merchant IDs: 0
Chargeback missing IDs: missing_complaint_id    0
missing_txn_id          0
missing_user_id         0
missing_merchant_id     0
dtype: int64


In [27]:
# ============================================================
# CELL 8A: KYC ENTITY CONFLICT DETECTION
# ============================================================

# Attributes that should normally remain consistent
# for the same user across KYC records.

kyc_attributes = [
    "full_name",
    "pan",
    "aadhaar",
    "date_of_birth",
    "city",
    "occupation",
    "kyc_status"
]


# Count distinct values for every attribute per user
kyc_conflict_summary = (
    kyc.groupby("user_id", dropna=True)[kyc_attributes]
    .nunique(dropna=True)
)


# A user is considered conflicting if any attribute
# has more than one distinct value.
kyc_conflict_summary["kyc_attribute_conflict"] = (
    kyc_conflict_summary[kyc_attributes] > 1
).any(axis=1)


# Map the conflict flag back to every KYC record
kyc["kyc_attribute_conflict"] = (
    kyc["user_id"]
    .map(kyc_conflict_summary["kyc_attribute_conflict"])
    .fillna(False)
)


print(
    "Users with conflicting KYC attributes:",
    kyc_conflict_summary["kyc_attribute_conflict"].sum()
)

Users with conflicting KYC attributes: 5860


In [28]:
# ============================================================
# CELL 8B: MERCHANT ENTITY CONFLICT DETECTION
# ============================================================

merchant_attributes = [
    "merchant_name",
    "merchant_category",
    "mcc",
    "city",
    "merchant_status"
]

merchant_conflict_summary = (
    merchants.groupby("merchant_id", dropna=True)[merchant_attributes]
    .nunique(dropna=True)
)

merchant_conflict_summary["merchant_attribute_conflict"] = (
    merchant_conflict_summary > 1
).any(axis=1)

merchants["merchant_attribute_conflict"] = (
    merchants["merchant_id"]
    .map(
        merchant_conflict_summary["merchant_attribute_conflict"]
    )
    .fillna(False)
)

print(
    "Merchants with conflicting attributes:",
    merchant_conflict_summary[
        "merchant_attribute_conflict"
    ].sum()
)

Merchants with conflicting attributes: 1358


In [29]:
# ============================================================
# CELL 8C: CHARGEBACK TRANSACTION-LEVEL SUMMARY
# ============================================================

# Define severity priority
severity_rank = {
    "LOW": 1,
    "MEDIUM": 2,
    "HIGH": 3,
    "CRITICAL": 4
}


# Create numeric severity rank
chargebacks["severity_rank"] = (
    chargebacks["severity"]
    .map(severity_rank)
)


# Aggregate chargebacks at transaction level
chargeback_txn_summary = (
    chargebacks
    .dropna(subset=["txn_id"])
    .groupby("txn_id")
    .agg(
        # Number of chargeback records for the transaction
        chargeback_count=("complaint_id", "count"),

        # Total disputed amount
        # min_count=1 prevents all-invalid values becoming 0
        chargeback_amount=(
            "disputed_amount",
            lambda x: x.sum(min_count=1)
        ),

        # Highest severity rank
        max_severity_rank=("severity_rank", "max"),

        # Number of open chargebacks
        open_chargebacks=(
            "resolution_status",
            lambda x: (x == "OPEN").sum()
        ),

        # Number of critical chargebacks
        critical_chargebacks=(
            "severity",
            lambda x: (x == "CRITICAL").sum()
        )
    )
    .reset_index()
)


# Convert maximum severity rank back to label
reverse_severity_rank = {
    1: "LOW",
    2: "MEDIUM",
    3: "HIGH",
    4: "CRITICAL"
}

chargeback_txn_summary["max_chargeback_severity"] = (
    chargeback_txn_summary["max_severity_rank"]
    .map(reverse_severity_rank)
)


# Remove helper column
chargeback_txn_summary.drop(
    columns=["max_severity_rank"],
    inplace=True
)


print(
    "Transactions with chargebacks:",
    len(chargeback_txn_summary)
)

print(
    "Total chargeback records included:",
    chargeback_txn_summary["chargeback_count"].sum()
)

print("\nChargeback summary preview:")
display(chargeback_txn_summary.head())

Transactions with chargebacks: 2568
Total chargeback records included: 2884

Chargeback summary preview:


,txn_id,chargeback_count,chargeback_amount,open_chargebacks,critical_chargebacks,max_chargeback_severity
0,TXN,81,172430.44,34,4,CRITICAL
1,TXN00000007,1,2231.72,1,0,HIGH
2,TXN00000019,1,377.11,1,0,HIGH
3,TXN00000021,2,2568.10,0,0,LOW
4,TXN00000025,1,1302.18,0,0,LOW


In [30]:
# ============================================================
# CELL 8D: ROW COUNT VALIDATION
# ============================================================

print("========== FINAL ROW COUNT CHECK ==========")

print("Transactions :", len(transactions))
print("KYC          :", len(kyc))
print("Merchants    :", len(merchants))
print("Chargebacks  :", len(chargebacks))

assert len(transactions) == 20400, "Transaction rows changed!"
assert len(kyc) == 36400, "KYC rows changed!"
assert len(merchants) == 6210, "Merchant rows changed!"
assert len(chargebacks) == 2884, "Chargeback rows changed!"

print("\n✓ All raw row counts preserved.")

========== FINAL ROW COUNT CHECK ==========
Transactions : 20400
KYC          : 36400
Merchants    : 6210
Chargebacks  : 2884

✓ All raw row counts preserved.


## 8. Export Processed Data

In [31]:
# ============================================================
# CELL 9: SAVE CLEANED DATASETS
# ============================================================

# Ensure processed directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# Save cleaned datasets
transactions.to_csv(
    PROCESSED_DIR / "upi_transactions_clean.csv",
    index=False
)

kyc.to_csv(
    PROCESSED_DIR / "kyc_clean.csv",
    index=False
)

merchants.to_csv(
    PROCESSED_DIR / "merchants_clean.csv",
    index=False
)

chargebacks.to_json(
    PROCESSED_DIR / "chargebacks_clean.json",
    orient="records",
    indent=4
)

chargebacks.to_csv(
    PROCESSED_DIR / "chargebacks_clean.csv",
    index=False
)

# Save transaction-level chargeback summary
chargeback_txn_summary.to_csv(
    PROCESSED_DIR / "chargeback_transaction_summary.csv",
    index=False
)

print("================================================")
print("DATA CLEANING PIPELINE COMPLETED SUCCESSFULLY")
print("================================================")

print(f"Transactions : {len(transactions):,}")
print(f"KYC          : {len(kyc):,}")
print(f"Merchants    : {len(merchants):,}")
print(f"Chargebacks  : {len(chargebacks):,}")

print("\nFiles saved in:")
print(PROCESSED_DIR)

DATA CLEANING PIPELINE COMPLETED SUCCESSFULLY
Transactions : 20,400
KYC          : 36,400
Merchants    : 6,210
Chargebacks  : 2,884

Files saved in:
..\data\processed


In [32]:
# ============================================================
# FINAL PROCESSED DATA VERIFICATION
# ============================================================

print("========== PROCESSED DATA VERIFICATION ==========")

print("Transactions :", len(transactions))
print("KYC          :", len(kyc))
print("Merchants    :", len(merchants))
print("Chargebacks  :", len(chargebacks))

print("\nFiles successfully written to:")
print(PROCESSED_DIR)

========== PROCESSED DATA VERIFICATION ==========
Transactions : 20400
KYC          : 36400
Merchants    : 6210
Chargebacks  : 2884

Files successfully written to:
..\data\processed
